In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

#sklearn classification models 
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB



#Pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler

#Dimensionality Reduction
from sklearn.decomposition import PCA

#Hyperparameter Tuning
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

#Evaluation Metrics F1, ROC OVR
# Balanced Accuracy,

# · The macro-average and micro-average of the One-vs-Rest ROC AUC score.

# In addition,

# · Draw the macro-average One-vs-Rest ROCs for all classification methods into a single graph to allow for easy comparison between methods.

# · Draw the micro-average One-vs-Rest ROCs for all classification methods into a single graph to allow for easy comparison between methods.

# · For each class, draw the One-vs-Rest ROCs for all classification methods into a separate graph, to allow for easy comparison between methods.
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

#Confusion Matrix
from sklearn.metrics import confusion_matrix





In [ ]:
#data load
from sklearn.datasets import fetch_openml

dataset = fetch_openml(data_id=4538, as_frame=False) # fetch the dataset


In [ ]:

X = dataset.data
y = dataset.target


print(X.shape)
print(y.shape)

#checking integrity of data
distribution = np.unique(y, return_counts=True)
isnan = np.isnan(X).sum()

print("general distribution of data :", distribution)
print("Nans in data :", isnan)


(9873, 32)
(9873,)
general distribution of data : (array(['D', 'H', 'P', 'R', 'S'], dtype=object), array([2741,  998, 2097, 1087, 2950]))
Nans in data : 0


In [36]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(
    contamination=0.05,
    random_state=42
)

outlier_labels = iso.fit_predict(X)
# -1 = outlier, 1 = inlier

n_outliers = (outlier_labels == -1).sum()
print("Detected outliers:", n_outliers)

Detected outliers: 494


In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, test_size=0.3, stratify=y, random_state=42) #keep data balanced

In [ ]:
#Model Selection without hyperparameter tuning

models = [
    LogisticRegression(),
    DecisionTreeClassifier(),
    RandomForestClassifier(),
    KNeighborsClassifier(),
    LinearSVC(),
    SVC(),
    GaussianNB()
]

model_names = [
    "Logistic Regression",
    "Decision Tree",
    "Random Forest",
    "KNN",
    "Linear SVC",
    "SVC",
    "Naive Bayes"
]

results = {}
for name, model in zip(model_names, models):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = balanced_accuracy_score(y_test, y_pred)
    results[name] = acc

print(results)

{'Logistic Regression': 0.30748896861726255, 'Decision Tree': 0.4578415326001883, 'Random Forest': 0.5898430876558634, 'KNN': 0.5581649197515796, 'Linear SVC': 0.3062805205741361, 'SVC': 0.4300664411211203, 'Naive Bayes': 0.3425543766949714}


In [ ]:
#hyperparameter tuning with default parameters

models = {
    "Logistic Regression": (
        Pipeline([
            ("scaler", RobustScaler()),
            ("clf", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "clf__C": [0.001, 0.01, 0.1, 1, 10,100]
        }
    ),
    "Logistic Regression (PCA)": (
        Pipeline([
            ("scaler", RobustScaler()),
            ("pca", PCA()),
            ("clf", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "pca__n_components": [5, 10, 15, 20, 25],
            "clf__C": [0.001, 0.01, 0.1, 1, 10,100]
        }
    ),

    "Decision Tree": (
        DecisionTreeClassifier(
            class_weight="balanced",
            random_state=42
        ),
        {
            "max_depth": [None, 10, 20, 30],
            "min_samples_leaf": [1, 5, 10]
        }
    ),

    "Random Forest": (
        RandomForestClassifier(
            class_weight="balanced",
            random_state=42
        ),
        {
            "n_estimators": [200, 500],
            "max_depth": [None, 15, 30],
            "min_samples_leaf": [1, 5]
        }
    ),

    "KNN": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier())
        ]),
        {
            "clf__n_neighbors": [3, 5, 7, 11],
            "clf__weights": ["uniform", "distance"]
        }
    ),

    "Linear SVC": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LinearSVC(
                class_weight="balanced",
                max_iter=5000
            ))
        ]),
        {
            "clf__C": [0.01, 0.1, 1, 10]
        }
    ),

    "SVC": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(
                class_weight="balanced",
                probability=True
            ))
        ]),
        {
            "clf__C": [0.1, 1, 10],
            "clf__gamma": ["scale", 0.01, 0.1]
        }
    ),

    "Naive Bayes": (
        GaussianNB(),
        {}
    )
}


tuned_results = {}
best_models = {}

for name, (model, params) in models.items():
    grid = GridSearchCV(
        model,
        params,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1
    )
    
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    
    acc = balanced_accuracy_score(y_test, y_pred)
    
    tuned_results[name] = acc
    best_models[name] = best_model

print("Tuned results:")
print(tuned_results)


Tuned results:
{'Logistic Regression': 0.43700762091970946, 'Logistic Regression (PCA)': 0.41905043484826415, 'Decision Tree': 0.472876417173258, 'Random Forest': 0.6045172736609855, 'KNN': 0.5645810025619401, 'Linear SVC': 0.40845816046638284, 'SVC': 0.5146802759973641, 'Naive Bayes': 0.3425543766949714}


## Logistic Regression

In [82]:
models_lr = {
    "Logistic Regression": (
        Pipeline([
            ("scaler", RobustScaler()),
            ("clf", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "clf__C": [0.001, 0.01, 0.1, 1, 10,100]
        }
    ),
    "Logistic Regression (PCA)": (
        Pipeline([
            ("scaler", RobustScaler()),
            ("pca", PCA()),
            ("clf", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "pca__n_components": [5, 10, 15, 20, 25],
            "clf__C": [0.001, 0.01, 0.1, 1, 10,100]
        }
    ),

}

tuned_results_lr = {}

for name, (model, params) in models_lr.items():
    search = GridSearchCV(
        model,
        params,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1
    )
    
    search.fit(X_train, y_train)
    
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)
    
    acc = balanced_accuracy_score(y_test, y_pred)
    
    tuned_results_lr[name] = acc


print("Tuned results:")
print(tuned_results_lr)


Tuned results:
{'Logistic Regression': 0.43700762091970946, 'Logistic Regression (PCA)': 0.41905043484826415}


## Decision Tree

In [83]:
models_dt = {
    "Decision Tree": (
        DecisionTreeClassifier(
            class_weight="balanced",
            random_state=42
        ),
        {
            "max_depth": [None, 10, 20, 30],
            "min_samples_leaf": [1, 5, 10]
        }
    ),
    
    "Decision Tree with Hyperparameters": (
        DecisionTreeClassifier(
            class_weight="balanced",
            random_state=42
        ),
        {
            "criterion": ["gini", "entropy", "log_loss"],
            "max_depth": [5, 10, 20, 40, None],
            "min_samples_split": [2, 5, 10, 20],
            "min_samples_leaf": [1, 2, 5, 10],
            "ccp_alpha": [0.0, 0.001, 0.01, 0.1]
        }
        
    ),
}

tuned_results_dt = {}
for name, (model, params) in models_dt.items():
    search = GridSearchCV(
        model,
        params,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1
    )
    search.fit(X_train, y_train)
    
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)
    
    acc = balanced_accuracy_score(y_test, y_pred)
    
    tuned_results_dt[name] = acc

print("Tuned results:")
print(tuned_results_dt)

Tuned results:
{'Decision Tree': 0.472876417173258, 'Decision Tree with Hyperparameters': 0.472876417173258}


## Random Forest

In [84]:
models_rf = {
    "Random Forest": (
        RandomForestClassifier(
            class_weight="balanced",
            random_state=42
        ),
        {
            "n_estimators": [200, 500],
            "max_depth": [None, 15, 30],
            "min_samples_leaf": [1, 5]
        }
    ),
    
   
}

tuned_results_rf = {}


for name, (model, params) in models_rf.items():
    grid = GridSearchCV(
        model,
        params,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1
    )
    
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    
    acc = balanced_accuracy_score(y_test, y_pred)
    
    tuned_results_rf[name] = acc

print("Tuned results:")
print(tuned_results_rf)


Tuned results:
{'Random Forest': 0.6045172736609855}


In [88]:
models_knn = {
      "KNN": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier())
        ]),
        {
            "clf__n_neighbors": [3, 5, 7, 11],
            "clf__weights": ["uniform", "distance"]
        }
    ),
}

tuned_results_knn = {}


for name, (model, params) in models_knn.items():
    grid = GridSearchCV(
        model,
        params,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1
    )
    
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    
    acc = balanced_accuracy_score(y_test, y_pred)
    
    tuned_results_knn[name] = acc

print("Tuned results:")
print(tuned_results_knn)


Tuned results:
{'KNN': 0.5645810025619401}


In [89]:
models_lsvc = {
    "Linear SVC": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LinearSVC(
                class_weight="balanced",
                max_iter=5000
            ))
        ]),
        {
            "clf__C": [0.01, 0.1, 1, 10]
        }
    ),
}

tuned_results_lsvc = {}


for name, (model, params) in models_lsvc.items():
    grid = GridSearchCV(
        model,
        params,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1
    )
    
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    
    acc = balanced_accuracy_score(y_test, y_pred)
    
    tuned_results_lsvc[name] = acc

print("Tuned results:")
print(tuned_results_lsvc)


Tuned results:
{'Linear SVC': 0.40845816046638284}


In [90]:
models_svc = {
    "SVC": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(
                class_weight="balanced",
                probability=True
            ))
        ]),
        {
            "clf__C": [0.1, 1, 10],
            "clf__gamma": ["scale", 0.01, 0.1]
        }
    ),
}

tuned_results_svc = {}


for name, (model, params) in models_svc.items():
    grid = GridSearchCV(
        model,
        params,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1
    )
    
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    
    acc = balanced_accuracy_score(y_test, y_pred)
    
    tuned_results_svc[name] = acc

print("Tuned results:")
print(tuned_results_svc)


Tuned results:
{'SVC': 0.5146802759973641}


In [91]:

models_nb = {
    "Naive Bayes": (
        GaussianNB(),
        {}
    )
}

tuned_results_nb = {}


for name, (model, params) in models_nb.items():
    grid = GridSearchCV(
        model,
        params,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1
    )
    
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    
    acc = balanced_accuracy_score(y_test, y_pred)
    
    tuned_results_nb[name] = acc

print("Tuned results:")
print(tuned_results_nb)


Tuned results:
{'Naive Bayes': 0.3425543766949714}


### Logistic Regression
{'Logistic Regression': 0.4349500231995328} with standard scalar
{'Logistic Regression': 0.43700762091970946} with robust scalar
{'Logistic Regression (PCA)': 0.41905043484826415} with robust scalar


### Grid Search `Decision`
{'Decision Tree': 0.472876417173258, 'Decision Tree with Hyperparameters': 0.472876417173258}

### Randomised Search CV `Decision`
{'Decision Tree': 0.472876417173258, 'Decision Tree with Hyperparameters': 0.472876417173258}

### Random Forest Randomised Search
{'Random Forest': 0.472876417173258}

### Random Forest Grid Search 
{'Random Forest': 0.6045172736609855}

### KNN Grid Search 
{'KNN': 0.5645810025619401}



